# Compare DRD2 docking scores in inactive and active receptor conformations

This notebook docks the five selected DRD2 hits and risperidone into two local DRD2 structures:

- **Inactive:** 6CM4_Na, the risperidone-bound inactive DRD2 structure used by the project.
- **Active:** 9BS9, the LSD-bound active DRD2 structure in complex with mini-GoA and scFv16.

Each ligand is prepared once and docked into both receptor structures using the same Vina scoring function, search settings, random seed, and cubic box size. Receptor-specific box centers come from each structure's crystallographic orthosteric-site ligand (risperidone in 6CM4; LSD in 9BS9). The notebook writes per-receptor results and an active-minus-inactive score comparison.

Vina scores are approximate scoring-function outputs, not experimental binding affinities or proof of agonism/antagonism. These receptor structures differ in conformation, construct, co-crystallized ligand, and resolved components, so score differences are hypothesis-generating.

## Scope and comparison design

The ligand source is `data/raw/5ht2c_docking_ligands.csv`, which contains five selected DRD2 hits plus Ritanserin and Risperidone. This notebook selects ranks 1–5 and Risperidone only; Ritanserin and all prediction/ranking tables are excluded.

The inactive receptor is the project's prepared 6CM4_Na model. The active receptor is extracted from local `docking/prepared/Active/9BS9.cif`, retaining the DRD2 chain and the receptor author-residue ranges used by the project for 6CM4. The 9BS9 LSD component and the local raw 6CM4 risperidone coordinates define the receptor-specific box centers. Both receptor runs use the same 24 Å cubic box, Vina settings, and ligand PDBQT files.

A more negative Vina score is more favorable under this scoring function. The active-minus-inactive delta is `score_active - score_inactive`; a negative delta favors the active model. Differences can reflect receptor preparation and structure-specific factors, including the sodium ion present in the inactive 6CM4_Na receptor but not modeled in 9BS9.

In [1]:
from __future__ import annotations

import csv
import math
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (
            (candidate / "data/raw/5ht2c_docking_ligands.csv").is_file()
            and (candidate / "data/processed/crystal_structures/6CM4_Na.mmcif").is_file()
            and (candidate / "docking/prepared/Active/9BS9.cif").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        f"Could not find the project root from {Path.cwd()}. "
        "Run this notebook from inside the repository."
    )


ROOT = find_project_root()
LIGANDS_CSV = ROOT / "data/raw/5ht2c_docking_ligands.csv"
INACTIVE_RECEPTOR_CIF = ROOT / "data/processed/crystal_structures/6CM4_Na.mmcif"
INACTIVE_NATIVE_LIGAND_CIF = ROOT / "data/raw/crystal_structures/6cm4.mmcif"
INACTIVE_RECEPTOR_PDBQT = ROOT / "docking/prepared/6CM4_Na/drd2.pdbqt"
ACTIVE_RECEPTOR_CIF = ROOT / "docking/prepared/Active/9BS9.cif"
ACTIVE_PREP_DIR = ROOT / "docking/prepared/Active/9BS9_DRD2"
RUN_DIR = ROOT / "docking/runs/drd2_active_inactive_top5"
RESULTS_DIR = ROOT / "results/tables"
RESULTS_CSV = RESULTS_DIR / "drd2_active_inactive_top5_vina_scores.csv"
COMPARISON_CSV = RESULTS_DIR / "drd2_active_inactive_top5_vina_comparison.csv"

INACTIVE_NATIVE_LIGAND_COMPONENT = "8NU"  # Co-crystallized risperidone in 6CM4
ACTIVE_NATIVE_LIGAND_COMPONENT = "7LD"  # Co-crystallized LSD in 9BS9
ACTIVE_RECEPTOR_AUTHOR_CHAIN = "R"
DRD2_AUTHOR_RESIDUE_RANGES = ((35, 222), (364, 442))
BOX_SIZE_ANGSTROM = 24.0
EXHAUSTIVENESS = 16
NUM_POSES = 10
RANDOM_SEED = 22


## Check the local docking environment

Run in the project's Linux/WSL docking environment with RDKit, Biopython, PDBFixer, OpenMM, Meeko, and AutoDock Vina installed.

In [2]:
import importlib.util


def require_dependencies() -> dict[str, str]:
    required_modules = (
        "Bio",
        "pdbfixer",
        "openmm",
        "rdkit",
        "vina",
    )
    missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
    if missing_modules:
        raise ImportError(
            "Missing docking packages: " + ", ".join(missing_modules)
            + ". Select the project's drd2-vina Python environment."
        )
    if sys.platform == "win32":
        raise RuntimeError("Run the docking notebook in Linux/WSL or another supported Unix environment.")

    tools = {}
    scripts_dir = Path(sysconfig.get_path("scripts"))
    for name in ("mk_prepare_receptor.py", "mk_prepare_ligand.py"):
        located = shutil.which(name) or (str(scripts_dir / name) if (scripts_dir / name).is_file() else None)
        if located is None:
            raise FileNotFoundError(f"Required Meeko executable not found: {name}")
        tools[name] = located
    return tools


TOOLS = require_dependencies()
for path in (LIGANDS_CSV, INACTIVE_RECEPTOR_CIF, INACTIVE_NATIVE_LIGAND_CIF,
             INACTIVE_RECEPTOR_PDBQT, ACTIVE_RECEPTOR_CIF):
    if not path.is_file():
        raise FileNotFoundError(f"Required local input is missing: {path}")
print("Docking tools and local active/inactive DRD2 inputs are available.")

Docking tools and local active/inactive DRD2 inputs are available.


## Prepare the active and inactive DRD2 receptors

The inactive 6CM4_Na receptor uses the project's existing prepared PDBQT, which includes its crystallographic sodium ion. The active 9BS9 receptor is extracted from author chain R and repaired with PDBFixer before Meeko conversion. Non-receptor chains, the LSD ligand, and other hetero components are excluded from the active receptor.

### Site definition

The box center for inactive 6CM4 comes from co-crystallized risperidone (`8NU`) in the local raw structure. The active 9BS9 box center comes from co-crystallized LSD (`7LD`). Both use the same 24 Å cubic box. This is a consistent search-box size, while the experimental ligand coordinates provide the center in each receptor's coordinate frame.

In [3]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from openmm.app import PDBFile
from pdbfixer import PDBFixer


def cif_column(cif: dict, name: str) -> list[str]:
    values = cif.get(name)
    if values is None:
        raise KeyError(f"Required mmCIF column is missing: {name}")
    return values if isinstance(values, list) else [values]


def native_ligand_center(cif_path: Path, component_id: str) -> list[float]:
    cif = MMCIF2Dict(str(cif_path))
    columns = {
        name: cif_column(cif, key)
        for name, key in {
            "record": "_atom_site.group_PDB",
            "component": "_atom_site.label_comp_id",
            "element": "_atom_site.type_symbol",
            "x": "_atom_site.Cartn_x",
            "y": "_atom_site.Cartn_y",
            "z": "_atom_site.Cartn_z",
        }.items()
    }
    count = len(columns["record"])
    if not count or any(len(values) != count for values in columns.values()):
        raise ValueError(f"Incomplete atom-site data in {cif_path}.")
    points = [
        tuple(float(columns[axis][i]) for axis in ("x", "y", "z"))
        for i in range(count)
        if columns["record"][i] == "HETATM"
        and columns["component"][i] == component_id
        and columns["element"][i].strip().upper() not in {"H", "D"}
    ]
    if not points:
        raise ValueError(f"No heavy atoms for native ligand {component_id} in {cif_path}.")
    return [(min(axis) + max(axis)) / 2 for axis in zip(*points)]


def extract_active_drd2(cif_path: Path, output_path: Path) -> int:
    cif = MMCIF2Dict(str(cif_path))
    columns = {
        name: cif_column(cif, key)
        for name, key in {
            "record": "_atom_site.group_PDB",
            "chain": "_atom_site.auth_asym_id",
            "sequence": "_atom_site.auth_seq_id",
            "component": "_atom_site.label_comp_id",
            "atom": "_atom_site.label_atom_id",
            "altloc": "_atom_site.label_alt_id",
            "insertion": "_atom_site.pdbx_PDB_ins_code",
            "x": "_atom_site.Cartn_x",
            "y": "_atom_site.Cartn_y",
            "z": "_atom_site.Cartn_z",
            "occupancy": "_atom_site.occupancy",
            "b_factor": "_atom_site.B_iso_or_equiv",
            "element": "_atom_site.type_symbol",
        }.items()
    }
    count = len(columns["record"])
    if not count or any(len(values) != count for values in columns.values()):
        raise ValueError(f"Incomplete atom-site data in {cif_path}.")

    alternate_occupancy: dict[tuple[str, str, str], dict[str, float]] = {}
    for i in range(count):
        if columns["record"][i] != "ATOM" or columns["chain"][i] != ACTIVE_RECEPTOR_AUTHOR_CHAIN:
            continue
        altloc = columns["altloc"][i]
        if altloc in (".", "?", " "):
            continue
        key = (columns["chain"][i], columns["sequence"][i], columns["insertion"][i])
        occupancies = alternate_occupancy.setdefault(key, {})
        occupancies[altloc] = occupancies.get(altloc, 0.0) + float(columns["occupancy"][i])
    preferred_altloc = {
        key: max(occupancies, key=occupancies.get)
        for key, occupancies in alternate_occupancy.items()
    }

    lines = ["REMARK DRD2 active receptor extracted from local 9BS9.cif\n"]
    residues = set()
    serial = 1
    for i in range(count):
        if columns["record"][i] != "ATOM" or columns["chain"][i] != ACTIVE_RECEPTOR_AUTHOR_CHAIN:
            continue
        try:
            residue_number = int(columns["sequence"][i])
        except ValueError:
            continue
        if not any(start <= residue_number <= end for start, end in DRD2_AUTHOR_RESIDUE_RANGES):
            continue
        insertion = columns["insertion"][i]
        key = (columns["chain"][i], columns["sequence"][i], insertion)
        altloc = columns["altloc"][i]
        if altloc not in (".", "?", " ") and altloc != preferred_altloc.get(key):
            continue
        component = columns["component"][i]
        atom = columns["atom"][i]
        element = columns["element"][i].strip().upper()
        insertion = " " if insertion in (".", "?", " ") else insertion
        atom_field = f"{atom:<4s}" if len(atom) == 4 or atom[:1].isdigit() else f" {atom:<3s}"
        x, y, z = (float(columns[axis][i]) for axis in ("x", "y", "z"))
        occupancy = float(columns["occupancy"][i])
        b_factor = float(columns["b_factor"][i])
        lines.append(
            f"ATOM  {serial:5d} {atom_field} {component:>3s} A{residue_number:4d}{insertion:1s}   "
            f"{x:8.3f}{y:8.3f}{z:8.3f}{occupancy:6.2f}{b_factor:6.2f}          {element:>2s}\n"
        )
        residues.add((residue_number, insertion))
        serial += 1
    if len(residues) < 150:
        raise ValueError(f"Only {len(residues)} DRD2 residues were extracted from active 9BS9.")
    lines.extend(("TER\n", "END\n"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("".join(lines), encoding="ascii")
    return len(residues)


inactive_center = native_ligand_center(INACTIVE_NATIVE_LIGAND_CIF, INACTIVE_NATIVE_LIGAND_COMPONENT)
active_center = native_ligand_center(ACTIVE_RECEPTOR_CIF, ACTIVE_NATIVE_LIGAND_COMPONENT)
active_pdb = ACTIVE_PREP_DIR / "drd2_active_clean.pdb"
active_repaired_pdb = ACTIVE_PREP_DIR / "drd2_active_repaired.pdb"
active_pdbqt = ACTIVE_PREP_DIR / "drd2_active.pdbqt"
def correct_terminal_oxygen_geometry(repaired_pdb: Path) -> int:
    lines = repaired_pdb.read_text(encoding="ascii").splitlines(keepends=True)
    residues: dict[tuple[str, str], dict[str, int]] = {}
    for index, line in enumerate(lines):
        if line.startswith(("ATOM  ", "HETATM")):
            key = (line[21], line[22:27])
            residues.setdefault(key, {})[line[12:16].strip()] = index

    corrected = 0
    for atoms in residues.values():
        if not all(name in atoms for name in ("CA", "C", "O", "OXT")):
            continue

        def coordinates(atom_name: str) -> tuple[float, float, float]:
            line = lines[atoms[atom_name]]
            return tuple(float(line[start:start + 8]) for start in (30, 38, 46))

        ca, carbonyl_c, oxygen, terminal_oxygen = (
            coordinates(name) for name in ("CA", "C", "O", "OXT")
        )
        if math.dist(ca, terminal_oxygen) >= 1.8:
            continue
        ca_vector = tuple(ca[i] - carbonyl_c[i] for i in range(3))
        oxygen_vector = tuple(oxygen[i] - carbonyl_c[i] for i in range(3))

        def unit(vector: tuple[float, float, float]) -> tuple[float, float, float]:
            magnitude = math.sqrt(sum(value * value for value in vector))
            if magnitude < 1e-8:
                raise ValueError("Could not repair degenerate terminal oxygen geometry.")
            return tuple(value / magnitude for value in vector)

        ca_direction, oxygen_direction = unit(ca_vector), unit(oxygen_vector)
        opposite_bisector = tuple(
            -(ca_direction[i] + oxygen_direction[i]) for i in range(3)
        )
        terminal_direction = unit(opposite_bisector)
        repaired_position = tuple(
            carbonyl_c[i] + 1.25 * terminal_direction[i] for i in range(3)
        )
        index = atoms["OXT"]
        lines[index] = (
            lines[index][:30]
            + "".join(f"{value:8.3f}" for value in repaired_position)
            + lines[index][54:]
        )
        corrected += 1
    if corrected:
        repaired_pdb.write_text("".join(lines), encoding="ascii")
    return corrected


active_residue_count = extract_active_drd2(ACTIVE_RECEPTOR_CIF, active_pdb)
fixer = PDBFixer(filename=str(active_pdb))
fixer.findMissingResidues()
fixer.missingResidues = {}
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
with active_repaired_pdb.open("w", encoding="ascii") as output_file:
    PDBFile.writeFile(fixer.topology, fixer.positions, output_file, keepIds=True)
corrected_oxygen_count = correct_terminal_oxygen_geometry(active_repaired_pdb)
if corrected_oxygen_count:
    print(f"Corrected terminal oxygen geometry in {corrected_oxygen_count} residue(s).")

subprocess.run([
    TOOLS["mk_prepare_receptor.py"],
    "--read_pdb", str(active_repaired_pdb),
    "--write_pdbqt", str(active_pdbqt),
    "--box_center", *(f"{value:.3f}" for value in active_center),
    "--box_size", *(f"{BOX_SIZE_ANGSTROM:.3f}" for _ in range(3)),
    "--delete_bad_res_from_box_radius", "8",
], check=True)
if not active_pdbqt.is_file() or active_pdbqt.stat().st_size == 0:
    raise RuntimeError(f"Meeko did not prepare the active receptor: {active_pdbqt}")

receptors = [
    {
        "state": "inactive",
        "pdb_id": "6CM4_Na",
        "pdbqt": INACTIVE_RECEPTOR_PDBQT,
        "center": inactive_center,
        "native_ligand": INACTIVE_NATIVE_LIGAND_COMPONENT,
    },
    {
        "state": "active",
        "pdb_id": "9BS9",
        "pdbqt": active_pdbqt,
        "center": active_center,
        "native_ligand": ACTIVE_NATIVE_LIGAND_COMPONENT,
    },
]
print(f"Active receptor extracted: {active_residue_count} DRD2 residues; chain {ACTIVE_RECEPTOR_AUTHOR_CHAIN}.")
print("Inactive box center:", inactive_center)
print("Active box center:", active_center)
print(f"Shared Vina box size: {BOX_SIZE_ANGSTROM:.1f} Å per side")

Corrected terminal oxygen geometry in 1 residue(s).


Template padding will be used.
 gasteiger charges will be read from template file 
 

Files written:
/mnt/c/Users/manup/Documents/Leiden Universiteit/LUMC/Advanced Computational Methods in Drug Discovery/Mini-Research/drd2-antagonist-ml-docking/docking/prepared/Active/9BS9_DRD2/drd2_active.pdbqt <-- static (i.e., rigid) receptor input file


Active receptor extracted: 261 DRD2 residues; chain R.
Inactive box center: [9.1765, 5.4704999999999995, -8.6645]
Active box center: [130.257, 114.439, 95.4925]
Shared Vina box size: 24.0 Å per side


## Select and prepare the six ligands

Selection is limited to CSV rows with `role=selected_hit` and ranks 1–5, plus the single Risperidone reference. Ritanserin is intentionally excluded. The six ligands are prepared once from their listed SMILES, so exactly the same ligand PDBQT is used against both receptor states.

### Ligand preparation

RDKit adds hydrogens, generates reproducible 3D conformers, and minimizes them with MMFF (or UFF when MMFF parameters are unavailable). Meeko converts the resulting SDF files to PDBQT.

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem

with LIGANDS_CSV.open(newline="", encoding="utf-8") as handle:
    source_ligands = list(csv.DictReader(handle))
required_columns = {"selection_rank", "compound_id", "compound_name", "smiles", "role"}
if not source_ligands or not required_columns.issubset(source_ligands[0]):
    raise ValueError(f"Expected CSV columns {sorted(required_columns)} in {LIGANDS_CSV}.")

selected_hits = sorted(
    (row for row in source_ligands if row["role"] == "selected_hit"),
    key=lambda row: int(row["selection_rank"]),
)
risperidone_rows = [
    row for row in source_ligands
    if row["role"] == "reference_ligand" and row["compound_name"].strip().lower() == "risperidone"
]
if len(selected_hits) != 5 or [int(row["selection_rank"]) for row in selected_hits] != [1, 2, 3, 4, 5]:
    raise ValueError("The ligand CSV must contain exactly selected-hit ranks 1 through 5.")
if len(risperidone_rows) != 1:
    raise ValueError(f"Expected one Risperidone reference row, found {len(risperidone_rows)}.")
ligands = selected_hits + risperidone_rows
if len({row["compound_id"] for row in ligands}) != 6:
    raise ValueError("The five selected hits and Risperidone must have unique compound IDs.")


def safe_name(text: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", text).strip("_")
    if not slug:
        raise ValueError(f"Cannot create a safe file name from {text!r}.")
    return slug


RUN_DIR.mkdir(parents=True, exist_ok=True)
for ligand in ligands:
    molecule = Chem.MolFromSmiles(ligand["smiles"])
    if molecule is None:
        raise ValueError(f"Invalid SMILES for {ligand['compound_name']}: {ligand['smiles']}")
    molecule.SetProp("_Name", ligand["compound_name"])
    molecule = Chem.AddHs(molecule)
    parameters = AllChem.ETKDGv3()
    parameters.randomSeed = RANDOM_SEED
    if AllChem.EmbedMolecule(molecule, parameters) != 0:
        raise ValueError(f"Could not generate 3D coordinates for {ligand['compound_name']}.")
    if AllChem.MMFFHasAllMoleculeParams(molecule):
        AllChem.MMFFOptimizeMolecule(molecule)
    elif AllChem.UFFHasAllMoleculeParams(molecule):
        AllChem.UFFOptimizeMolecule(molecule)
    else:
        raise ValueError(f"No MMFF or UFF parameters for {ligand['compound_name']}.")

    ligand["run_name"] = safe_name(ligand["compound_id"])
    ligand_dir = RUN_DIR / "ligands" / ligand["run_name"]
    ligand_dir.mkdir(parents=True, exist_ok=True)
    ligand["sdf_path"] = ligand_dir / f"{ligand['run_name']}_3d.sdf"
    ligand["pdbqt_path"] = ligand_dir / f"{ligand['run_name']}.pdbqt"
    writer = Chem.SDWriter(str(ligand["sdf_path"]))
    writer.write(molecule)
    writer.close()
    subprocess.run([
        TOOLS["mk_prepare_ligand.py"],
        "-i", str(ligand["sdf_path"]),
        "-o", str(ligand["pdbqt_path"]),
    ], check=True)
    if not ligand["pdbqt_path"].is_file() or ligand["pdbqt_path"].stat().st_size == 0:
        raise RuntimeError(f"Meeko did not prepare {ligand['compound_name']} as PDBQT.")
    print(f"Prepared {ligand['compound_name']} ({ligand['compound_id']})")


Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


Prepared Top hit 1 (s_2230__9292524__16949512)


Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


Prepared Top hit 2 (s_2230__22865356__16949512)


Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


...

Prepared Top hit 4 (s_2230__14793138__7471562)


Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


Prepared Top hit 5 (s_2230__14793138__23291970)


Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


Prepared Risperidone (RISPERIDONE)


## Dock each ligand in both DRD2 conformations

Each of the six prepared ligands is docked once to each receptor. The raw CSV stores receptor-specific best scores and pose paths; the comparison CSV places active and inactive scores side-by-side and calculates `active - inactive` for each ligand.

### Run Vina and compare receptor-state scores

Scores are reported in kcal/mol. More negative is more favorable according to Vina's scoring function; do not interpret them as measured affinity.

In [5]:
from contextlib import redirect_stdout
from vina import Vina


def best_pose_score(poses_path: Path) -> float:
    for line in poses_path.read_text(encoding="utf-8").splitlines():
        if line.startswith("REMARK VINA RESULT:"):
            score = float(line.split()[3])
            if math.isfinite(score):
                return score
            raise ValueError(f"Non-finite Vina score in {poses_path}.")
    raise ValueError(f"No Vina score found in pose file {poses_path}.")


results: list[dict[str, str | int | float]] = []
for ligand in ligands:
    for receptor in receptors:
        run_folder = RUN_DIR / receptor["state"] / ligand["run_name"]
        run_folder.mkdir(parents=True, exist_ok=True)
        poses_path = run_folder / f"{ligand['run_name']}_poses.pdbqt"
        log_path = run_folder / "vina.log"
        vina = Vina(sf_name="vina", cpu=0, seed=RANDOM_SEED)
        vina.set_receptor(str(receptor["pdbqt"]))
        vina.set_ligand_from_file(str(ligand["pdbqt_path"]))
        vina.compute_vina_maps(
            center=receptor["center"],
            box_size=[BOX_SIZE_ANGSTROM] * 3,
        )
        with log_path.open("w", encoding="utf-8") as log_handle:
            with redirect_stdout(log_handle):
                vina.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=NUM_POSES)
                vina.write_poses(str(poses_path), n_poses=NUM_POSES, overwrite=True)
        score = best_pose_score(poses_path)
        results.append({
            "compound_id": ligand["compound_id"],
            "compound_name": ligand["compound_name"],
            "role": ligand["role"],
            "selection_rank": ligand["selection_rank"],
            "receptor_state": receptor["state"],
            "receptor_pdb_id": receptor["pdb_id"],
            "native_site_ligand_component": receptor["native_ligand"],
            "vina_score_kcal_mol": score,
            "poses_file": str(poses_path.relative_to(ROOT)),
        })
        print(f"{ligand['compound_name']} / {receptor['pdb_id']}: {score:.3f} kcal/mol")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
result_fields = [
    "compound_id", "compound_name", "role", "selection_rank", "receptor_state",
    "receptor_pdb_id", "native_site_ligand_component", "vina_score_kcal_mol", "poses_file",
]
with RESULTS_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=result_fields)
    writer.writeheader()
    writer.writerows(results)

scores = {(row["compound_id"], row["receptor_state"]): float(row["vina_score_kcal_mol"])
          for row in results}
comparison_rows = []
for ligand in ligands:
    inactive_score = scores[(ligand["compound_id"], "inactive")]
    active_score = scores[(ligand["compound_id"], "active")]
    comparison_rows.append({
        "compound_id": ligand["compound_id"],
        "compound_name": ligand["compound_name"],
        "role": ligand["role"],
        "selection_rank": ligand["selection_rank"],
        "inactive_6CM4_Na_vina_score_kcal_mol": inactive_score,
        "active_9BS9_vina_score_kcal_mol": active_score,
        "active_minus_inactive_kcal_mol": round(active_score - inactive_score, 3),
    })
comparison_fields = [
    "compound_id", "compound_name", "role", "selection_rank",
    "inactive_6CM4_Na_vina_score_kcal_mol", "active_9BS9_vina_score_kcal_mol",
    "active_minus_inactive_kcal_mol",
]
with COMPARISON_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=comparison_fields)
    writer.writeheader()
    writer.writerows(comparison_rows)

print(f"\nSaved {len(results)} receptor-specific scores to {RESULTS_CSV}")
print(f"Saved {len(comparison_rows)} active/inactive comparisons to {COMPARISON_CSV}")
print("A negative active-minus-inactive delta means the active structure scored more favorably.")
for row in comparison_rows:
    print(
        f"{row['compound_name']:<12} "
        f"inactive={row['inactive_6CM4_Na_vina_score_kcal_mol']:.3f} "
        f"active={row['active_9BS9_vina_score_kcal_mol']:.3f} "
        f"delta={row['active_minus_inactive_kcal_mol']:+.3f} kcal/mol"
    )
print("These Vina scores are computational estimates, not experimental binding affinities.")

Computing Vina grid ... done.


Performing docking (random seed: 22) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Top hit 1 / 6CM4_Na: -10.797 kcal/mol



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1        -10.8          0          0
   2        -10.4      1.225      2.141
   3       -9.994      2.024       3.38
   4       -9.824      4.079      9.554
   5       -9.822      3.724      9.472
   6       -9.687      1.894      2.948
   7       -9.539      4.316      9.536
   8        -9.46      4.736      10.33
   9       -9.369      2.033      3.093
  10       -9.177      4.857      10.48
Computing Vina grid ... done.


Performing docking (random seed: 22) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Top hit 1 / 9BS9: -9.113 kcal/mol



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -9.113          0          0
   2       -9.046      3.356      8.703
   3       -8.714      3.902      8.923
   4       -8.602      4.052      9.973
   5       -8.518      1.589      2.498
   6        -8.49      3.577      9.402
   7       -8.448      3.497      9.561
   8       -8.442      4.181      10.15
   9        -8.29      4.053      10.06
  10        -8.28      4.379      9.846
Computing Vina grid ... done.


...

Performing docking (random seed: 22) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Top hit 5 / 9BS9: -9.649 kcal/mol



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -9.649          0          0
   2       -9.366      1.113      1.908
   3       -8.806      4.046      9.636
   4       -8.722      4.096      9.822
   5       -8.684      2.354      3.351
   6       -8.668      1.748      2.463
   7       -8.528      1.748      2.287
   8       -8.497      1.401       2.24
   9       -8.266      4.005      6.913
  10        -8.19      3.621      6.333
Computing Vina grid ... done.


Performing docking (random seed: 22) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Risperidone / 6CM4_Na: -11.220 kcal/mol



mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -11.22          0          0
   2       -11.14      1.345      2.197
   3       -10.99      3.308      9.822
   4        -10.8      3.916      10.34
   5       -10.72      2.839      5.174
   6       -10.58      4.497      11.11
   7       -10.47       4.24      10.93
   8       -10.47       3.72      10.16
   9        -10.3      2.543      4.196
  10       -9.969      4.233      10.87
Computing Vina grid ... done.


Performing docking (random seed: 22) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Risperidone / 9BS9: -10.001 kcal/mol

Saved 12 receptor-specific scores to /mnt/c/Users/manup/Documents/Leiden Universiteit/LUMC/Advanced Computational Methods in Drug Discovery/Mini-Research/drd2-antagonist-ml-docking/results/tables/drd2_active_inactive_top5_vina_scores.csv
Saved 6 active/inactive comparisons to /mnt/c/Users/manup/Documents/Leiden Universiteit/LUMC/Advanced Computational Methods in Drug Discovery/Mini-Research/drd2-antagonist-ml-docking/results/tables/drd2_active_inactive_top5_vina_comparison.csv
A negative active-minus-inactive delta means the active structure scored more favorably.
Top hit 1    inactive=-10.797 active=-9.113 delta=+1.684 kcal/mol
Top hit 2    inactive=-11.551 active=-9.705 delta=+1.846 kcal/mol
Top hit 3    inactive=-11.056 active=-9.576 delta=+1.480 k